In [ ]:
pip install gamspy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 223.8/223.8 kB 3.3 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 22.0 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 33.6 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 463.6/463.6 kB 23.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.0/47.0 kB 3.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 25.3 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.6/44.6 kB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 243.4/243.4 kB 18.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.3/87.3 kB 5.4 MB/s eta 0:00:00
  Attempting uninstall: typing-extensions
    Found existing installation: typing_extensions 4.7.1
    Uninstalling typing_extensions-4.7.1:
      Successfully uninstalled typing_extensions-4.7.1
Note: you may need to restart the kernel to u

In [1]:
# Import relevant packages
import gamspy as gp
# from gamspy import Container
# from gams import transfer
# import gams.transfer as gt
# import gdxpds
import numpy as np
import pandas as pd
from pathlib import Path
import re
import os

In [19]:
os.getcwd()

'/Users/ahanamukherjee/seer/seer'

### Data Pre-Processing & Exploration

In [2]:
# Define a function to load all .gdx files 
def load_gdx_files_simple(data_dir): 

    # Initialize an empty dictionary to store the data 
    all_data = {}
    for country_dir in Path(data_dir).iterdir(): 
        gdx_file = country_dir / 'input.gdx' 
        if gdx_file.exists():
            m = gp.Container() 
            m.read(str(gdx_file)) 
            # Convert all symbols into dataframes 
            country_data = {}
            for symbol_name in m.listSymbols(): 
                country_data[symbol_name] = pd.DataFrame(m[symbol_name].records) 

            all_data[country_dir.name] = country_data 
    
    return all_data

In [3]:
# Load all of the gdx files into a dictionary
all_data_simple = load_gdx_files_simple("data/raw")

In [4]:
# Check if all of the symbol names are the same across all countries
country_symbols = {country: set(symbols.keys()) for country, symbols in all_data_simple.items()}
symbol_check = all(s == list(country_symbols.values())[0] for s in country_symbols.values())
print(f'Does all country data use the same set of symbols? {symbol_check}')
print("")

# Find the "common" symbols across all countries
common_symbols = set.intersection(*country_symbols.values())
print("Common symbols in all countries:")
print(common_symbols)
print("")

# Find country-specific differences
for country, symbols in country_symbols.items():
    unique_to_country = symbols - common_symbols
    if unique_to_country:
        print(f"Symbols unique to {country}: {unique_to_country}")

Does all country data use the same set of symbols? False

Common symbols in all countries:
{'pEmissionsTotal', 'pCSPData', 'pVREProfile', 'pEnergyEfficiencyFactor', 'pAvailability', 'ftfindex', 'pCarbonPrice', 'pDemandData', 'pStorDataExcel', 'y', 'pFuelPrice', 'pTransferLimit', 'pVREgenProfile', 'pDemandForecast', 'pNewTransmission', 'pDemandProfile', 'pTradePrice', 'pHours', 'pMaxFuellimit'}

Symbols unique to Eswatini: {'zcmapExcel', 'pScalars', 'pPlanningReserveMargin', 'pZoneIndex', 'pCapexTrajectoryH2', 'pTechDataExcel', 'pLossFactor', 'Relevant', 'pEmissionsCountry', 'pExternalH2', 'pFuelTypeCarbonContent', 'MapGG', 'pSpinningReserveReqSystem', 'hh', 'sTopology', 'pSpinningReserveReqCountry', 'pH2DataExcel', 'pMaxExchangeShareTotal', 'pAvailabilityH2', 'peak', 'pExtTransferLimit', 'zext', 'pMaxExchangeShare', 'pFuelData', 'pCapexTrajectory', 'pGenDataExcel'}
Symbols unique to Tanzania: {'zcmapExcel', 'pAnnualMaxBuild', 'pScalars', 'pPlanningReserveMargin', 'pZoneIndex', 'pCapexT

In [5]:
for symbol in common_symbols:
    print(f"\nChecking symbol: {symbol}")
    
    base_country = list(all_data_simple.keys())[0]
    base_cols = set(all_data_simple[base_country][symbol].columns)
    
    for country in all_data_simple:
        cols = set(all_data_simple[country][symbol].columns)
        
        if cols != base_cols:
            print(f"  Differences in {country}:")
            print(f"    Missing: {base_cols - cols}")
            print(f"    Extra:   {cols - base_cols}")


Checking symbol: pEmissionsTotal
  Differences in EAPP:
    Missing: set()
    Extra:   {'value', 'y'}
  Differences in Malawi:
    Missing: set()
    Extra:   {'value', 'uni'}
  Differences in WAPP:
    Missing: set()
    Extra:   {'value', 'uni'}
  Differences in Guinea:
    Missing: set()
    Extra:   {'value', 'uni'}
  Differences in Rwanda:
    Missing: set()
    Extra:   {'value', 'y'}
  Differences in Turkiye:
    Missing: set()
    Extra:   {'value', 'uni'}

Checking symbol: pCSPData
  Differences in Tanzania:
    Missing: set()
    Extra:   {'value', 'uni_2', 'uni_0', 'uni_1'}
  Differences in Malawi:
    Missing: set()
    Extra:   {'value', 'uni_2', 'uni_0', 'uni_1'}
  Differences in WAPP:
    Missing: set()
    Extra:   {'value', 'uni_2', 'uni_0', 'uni_1'}
  Differences in Rwanda:
    Missing: set()
    Extra:   {'g', 'value', 'csphrd', 'shdr'}
  Differences in Turkiye:
    Missing: set()
    Extra:   {'value', 'uni_2', 'uni_0', 'uni_1'}

Checking symbol: pVREProfile
  Dif

In [6]:
# Obtain all of the symbols used across dataset
all_symbols = list(set().union(*country_symbols.values()))
all_symbols.sort()

# Convert to a Pandas series and output to a CSV
all_symbols = pd.Series(all_symbols)
all_symbols.to_csv('data/processed/all_symbols.csv', index=False)

### Input File Processing

In [7]:
# Define a function to build the symbol mapping schema
def build_symbol_registry(mapping_path):

    # Read in the CSV file for symbol mapping
    mapping_df = pd.read_csv(mapping_path)

    # Eliminate any whitespaces in names
    for col in mapping_df.select_dtypes(include='object'):
        mapping_df[col] = mapping_df[col].str.strip()

    # Extract the column names for each symbol
    col_names = [c for c in mapping_df.columns if c.startswith('col_')]

    # Initilize empty dictionary to store symbol mapping
    symbol_registry = {}

    # Build the dictionary
    for _, row in mapping_df.iterrows():

        # Define the raw and standardized symbol name
        resolved = row['canonical']
        raw_name = row['raw']

        # Define the cols
        cols = [row[c] for c in col_names if pd.notna(row[c]) and row[c] != ""]

        # Initialize a new canonical symbol if it doesn't exist in dictionary
        if resolved not in symbol_registry:
            symbol_registry[resolved] = {
                'raw_names': [],
                'columns': cols
            }

        # Append raw name aliases to the canonical symbol
        symbol_registry[resolved]['raw_names'].append(raw_name)

    return symbol_registry


# Define a function to look up the raw symbol name in the symbol registry
def symbol_lookup(raw_symbol, registry):

    for resolved_name, dict_items in registry.items():
        if raw_symbol in dict_items['raw_names']:
            return resolved_name
        
    return None

In [21]:
# Define a function infer column names for symbol data
def infer_names(col_series):
    """Attempt to infer standardized column name from column value types"""

    # Extract a sample of data from the column for text analysis
    sample = col_series.dropna().astype(str).head(50)

    if sample.empty:
        return None
    
    # Hour (t01, t1, t24, etc)
    if sample.str.match(r"^t\d+$", case=False).all():
        return "hour"
    
    # Year detection (4-digit years)
    if sample.str.match(r"^\d{4}$").all():
        return "year"

    # Season detection (dry/wet, Q1, m1, etc.)
    if sample.str.match(r"^(dry|wet|Q\d+|m\d+)$", case=False).all():
        return "season"

    # Data detecton (d1, d2, d3)
    if sample.str.match(r"^d\d+$", case=False).all():
        return "day"

    # Day detection (small integers)
    if sample.str.match(r"^\d+$").all():
        max_val = sample.astype(int).max()
        if max_val <= 31:
            return "day"
    
    # Parameter column detection
    # Heuristic: mostly alphabetic labels
    text_ratio = sample.str.contains(r"[A-Za-z]").mean()
    if text_ratio > 0.8:
        return "parameters"

    return None

In [ ]:
"""CURRENTLY NOT WORKING PROPERLY - CONSIDER REFACTORING INTO CLASS"""

# Define a function to rename symbols according to symbol registry and update parameter column headers
def symbol_sync(df, symbol_obj, resolved_name, registry, strict=False):

    # Check in the resolved symbol name exists in the registry
    if resolved_name not in registry:
        return df
    
    # Extract the expected column headers for the symbol
    expected_cols = registry[resolved_name].get("columns", [])
    expected_dims = [c for c in expected_cols if c != 'value']

    # If no column names are defined, return the df
    if not expected_cols:
        return df
    
    # Create a copy of the dataframe for renaming columns
    df = df.copy()

    # Obtain the number of dimensions of the symbol parameter
    domain_count = len(symbol_obj.domain)
    

    # --------------------------------------------------
    # 1. Rename value column and infer column names based on data types
    # --------------------------------------------------
    rename_map = {}
    used_labels = set()

    for col in df.columns:

        # Keep numeric value columns
        if col.lower() == 'value':
            rename_map[col] = 'value'
            used_labels.add('value')
            continue

        # Infer column names of other data types
        inferred = infer_names(df[col])

        if inferred and inferred not in used_labels:
            rename_map[col] = inferred
            used_labels.add(inferred)

    # Rename dataframe columns based on inferred rename map
    df = df.rename(columns=rename_map)

    # --------------------------------------------------
    # 2. Rename residual columns based on expected columns
    # --------------------------------------------------
    # Obtain the set of current column names in the dataframe
    current_cols = set(df.columns)

    # Extract the list of missing column names
    missing_expected = [
        col for col in expected_cols
        if col not in current_cols
    ]

    # Extract the list of expected column names
    unmapped_cols = [
        col for col in df.columns
        if col not in expected_cols and col != "value"
    ]

    # Assign leftover categorical columns to leftover expected dimensions
    for col, target in zip(unmapped_cols, missing_expected):
        if not pd.api.types.is_numeric_dtype(df[col]):
            df = df.rename(columns={col: target})

    # --------------------------------------------------
    # 4. Issue warning statement if there are any unresolved column names
    # --------------------------------------------------
    if strict:
        unresolved_cols = [
            col for col in df.columns 
            if col not in expected_cols
            ]
        if unresolved_cols:
            print(f"WARNING: {resolved_name} has unmapped columns after schema alignment: {unresolved_cols}")

    # --------------------------------------------------
    # 3. Fill in any non-matched columns with NaN
    # --------------------------------------------------
    for col in expected_cols:
        if col not in df.columns:
            df[col] = np.nan

    # Final ordering of columns
    df = df[expected_cols]

    return df

In [ ]:
"""CURRENTLY NOT WORKING PROPERLY - CONSIDER REFACTORING INTO CLASS"""

def symbol_sync(df, symbol_obj, resolved_name, registry, strict=False):
    """
    Map dataframe columns to canonical schema:
    - Last column is treated as 'value'
    - Parameters column detected via low-cardinality text heuristic
    - Remaining observed dims mapped to canonical schema in order
    - Extra canonical columns filled with NaN
    - Domain metadata used only for structural sanity check
    """

    expected_cols = registry[resolved_name]["columns"]

    # -------------------------------
    # 0. Handle empty dataframe
    # -------------------------------
    if df.empty or df.shape[1] == 0:
        # Return empty dataframe with canonical columns
        empty_df = pd.DataFrame(columns=expected_cols)
        return empty_df

    df = df.copy()
    expected_dims = [c for c in expected_cols if c != "value"]

    # -------------------------------
    # 1. Rename last column as 'value'
    # -------------------------------
    df = df.rename(columns={df.columns[-1]: "value"})
    used_labels = set(['value'])

    # -------------------------------
    # 2. Detect parameters column inline
    # -------------------------------
    # Low-cardinality text heuristic
    dim_candidates = [c for c in df.columns if c not in used_labels]
    parameters_col = None
    min_cardinality = None

    for col in dim_candidates:
        if df[col].dtype == object or pd.api.types.is_string_dtype(df[col]):
            cardinality = df[col].nunique(dropna=True)
            if min_cardinality is None or cardinality < min_cardinality:
                min_cardinality = cardinality
                parameters_col = col

    if parameters_col:
        df = df.rename(columns={parameters_col: 'parameters'})
        used_labels.add('parameters')

    # -------------------------------
    # 3. Structural sanity check using domain count
    # -------------------------------
    domain_count = len(symbol_obj.domain)
    dim_cols = [c for c in df.columns if c not in used_labels]

    if strict and len(dim_cols) > domain_count:
        print(
            f"WARNING: {resolved_name} has more observed dims ({len(dim_cols)}) "
            f"than symbol domain ({domain_count})"
        )

    # -------------------------------
    # 4. Map remaining observed dims to canonical schema
    # -------------------------------
    # Identify remaining unmapped observed columns
    remaining_cols = [c for c in df.columns if c not in used_labels and c != 'value']

    # Identify canonical dimensions still missing
    remaining_expected = [c for c in expected_dims if c not in used_labels]

    # Map by semantic priority instead of raw zip
    for col in remaining_cols:
        if 'zone' in remaining_expected:
            df = df.rename(columns={col: 'zone'})
            used_labels.add('zone')
            remaining_expected.remove('zone')
        elif remaining_expected:
            df = df.rename(columns={col: remaining_expected[0]})
            used_labels.add(remaining_expected[0])
            remaining_expected.pop(0)

    # -------------------------------
    # 5. Add missing canonical columns as NaN
    # -------------------------------
    for col in expected_cols:
        if col not in df.columns:
            df[col] = np.nan

    # -------------------------------
    # 6. Strict validation
    # -------------------------------
    if strict:
        unexpected = [c for c in df.columns if c not in expected_cols]
        if unexpected:
            print(f"WARNING: {resolved_name} has unexpected columns: {unexpected}")

    # -------------------------------
    # 7. Reorder according to canonical schema
    # -------------------------------
    # df = df[expected_cols]

    return df

In [60]:
# Define a function to load all .gdx files 
def load_gdx_files(data_dir, mapping_path, strict): 

    # Build the symbol mapping registry
    registry = build_symbol_registry(mapping_path)

    # Initialize an empty dictionary to store the data 
    all_data = {}
    for country_dir in Path(data_dir).iterdir(): 
        gdx_file = country_dir / 'input.gdx' 
        if gdx_file.exists():

            print(f'Processing {country_dir.name}')

            m = gp.Container() 
            m.read(str(gdx_file)) 

            # Convert all symbols into dataframes 
            country_data = {}

            for raw_symbol in m.listSymbols(): 

                # Find the corresponding resolved symbol name in the symbol registry
                resolved_symbol = symbol_lookup(raw_symbol, registry)

                if resolved_symbol is None:
                    continue
                
                # Convert the symbol records into a dataframe
                symbol_obj = m[raw_symbol]
                df = pd.DataFrame(symbol_obj.records)

                # Rename columns of the symbol data
                df = symbol_sync(df, symbol_obj, resolved_symbol, registry, strict)
                
                # Append cleaned df to country
                country_data[resolved_symbol] = df

            all_data[country_dir.name] = country_data 
    
    return all_data

In [61]:
# Load all of the gdx files into a dictionary
all_data = load_gdx_files(data_dir="data/raw", 
                          mapping_path="references/symbol_mapping.csv", 
                          strict=True)

Processing Eswatini
Processing Tanzania
Processing EAPP
Processing Malawi
Processing WAPP
Processing Serbia
Processing Timor Leste
Processing Gambia
Processing Guinea
Processing CAPP
Processing Rwanda
Processing Turkiye


### Accessing Data for PyPSA Input

In [57]:
all_data_simple['Tanzania']['pDemandProfile']

,uni_0,uni_1,uni_2,uni_3,value
0,Tanzania,Q1,d1,t01,0.519310
1,Tanzania,Q1,d1,t02,0.470345
2,Tanzania,Q1,d1,t03,0.440000
3,Tanzania,Q1,d1,t04,0.440000
4,Tanzania,Q1,d1,t05,0.440000
...,...,...,...,...,...
571,Tanzania,Q4,d6,t20,0.780000
572,Tanzania,Q4,d6,t21,0.744828
573,Tanzania,Q4,d6,t22,0.744828
574,Tanzania,Q4,d6,t23,0.686207


In [66]:
all_data['Tanzania']['pDemandProfile']

,zone,season,day,hour,value
0,Q1,d1,t01,NaN,0.519310
1,Q1,d1,t02,NaN,0.470345
2,Q1,d1,t03,NaN,0.440000
3,Q1,d1,t04,NaN,0.440000
4,Q1,d1,t05,NaN,0.440000
...,...,...,...,...,...
571,Q4,d6,t20,NaN,0.780000
572,Q4,d6,t21,NaN,0.744828
573,Q4,d6,t22,NaN,0.744828
574,Q4,d6,t23,NaN,0.686207


In [56]:
all_data_simple['EAPP']['pGenDataExcel']

,g,z,tech,f,uni,value
0,South,Djibouti,ICE,Diesel,Capacity,4.00
1,South,Djibouti,ICE,Diesel,FOMperMW,8000.00
2,South,Djibouti,ICE,Diesel,MinLimitShare,0.06
3,South,Djibouti,ICE,Diesel,RampDnRate,1.00
4,South,Djibouti,ICE,Diesel,RampUpRate,1.00
...,...,...,...,...,...,...
11890,Methane_generic,Rwanda,Methane,Methane,Life,25.00
11891,Methane_generic,Rwanda,Methane,Methane,HeatRate,9.20
11892,Methane_generic,Rwanda,Methane,Methane,VOM,4.50
11893,Methane_generic,Rwanda,Methane,Methane,Status,3.00


In [63]:
all_data['EAPP']['pGenDataInputCustom']

,generator,zone,tech,fuel,parameters,value
0,ICE,South,Diesel,Capacity,Djibouti,4.00
1,ICE,South,Diesel,FOMperMW,Djibouti,8000.00
2,ICE,South,Diesel,MinLimitShare,Djibouti,0.06
3,ICE,South,Diesel,RampDnRate,Djibouti,1.00
4,ICE,South,Diesel,RampUpRate,Djibouti,1.00
...,...,...,...,...,...,...
11890,Methane,Methane_generic,Methane,Life,Rwanda,25.00
11891,Methane,Methane_generic,Methane,HeatRate,Rwanda,9.20
11892,Methane,Methane_generic,Methane,VOM,Rwanda,4.50
11893,Methane,Methane_generic,Methane,Status,Rwanda,3.00


In [64]:
all_data['Eswatini']['pGenDataInputCustom']

,generator,zone,tech,fuel,parameters,value
0,NaN,Edwaleni1_HY_SWZ,NaN,NaN,StYr,1964.000
1,NaN,Edwaleni1_HY_SWZ,NaN,NaN,RetrYr,2050.000
2,NaN,Edwaleni1_HY_SWZ,NaN,NaN,Zone,1.000
3,NaN,Edwaleni1_HY_SWZ,NaN,NaN,Capacity,10.000
4,NaN,Edwaleni1_HY_SWZ,NaN,NaN,RampUpRate,1.000
...,...,...,...,...,...,...
663,NaN,SAPP_Connection_New,NaN,NaN,Type,8.000
664,NaN,SAPP_Connection_New,NaN,NaN,fuel1,6.000
665,NaN,SAPP_Connection_New,NaN,NaN,Status,3.000
666,NaN,SAPP_Connection_New,NaN,NaN,CapacityCredit,1.000


In [65]:
all_data_simple['Eswatini']['pGenDataExcel']

,uni_0,uni_1,value
0,Edwaleni1_HY_SWZ,StYr,1964.000
1,Edwaleni1_HY_SWZ,RetrYr,2050.000
2,Edwaleni1_HY_SWZ,Zone,1.000
3,Edwaleni1_HY_SWZ,Capacity,10.000
4,Edwaleni1_HY_SWZ,RampUpRate,1.000
...,...,...,...
663,SAPP_Connection_New,Type,8.000
664,SAPP_Connection_New,fuel1,6.000
665,SAPP_Connection_New,Status,3.000
666,SAPP_Connection_New,CapacityCredit,1.000
